# Entrenar un world model, ver cómo aprende y evaluarlo

Este notebook recorre el ciclo completo con **un** modelo: entrenarlo, mirar su curva de aprendizaje, medir su cross-entropy (CE), generar rollouts y compararlos con los de referencia, ver cómo manejan sus videos y medir la latencia. Es el mismo procedimiento que usan `scripts/train_all.sh` y `scripts/eval_all.sh` para el informe (`Docs/resultados.md`), pero con un tamaño de corrida que se elige acá.

Sirve en una GPU NVIDIA local (probado con 12 GB) y en Colab (*Entorno de ejecución > Cambiar tipo > T4*). Con los parámetros por defecto (`TOKENS = 20e6`, `RAPIDO = True`) es una demo de unos pocos minutos; el informe usa `100e6` tokens y la evaluación completa.

## 1. Preparación

Busca la raíz del repo (la carpeta que tiene `src/common.py`) y entra ahí. En Colab, si el repo no está, lo clona, instala las dependencias y baja los datos igual que `notebooks/colab.ipynb`. En local, si faltan los datos hay que correr `python -m src.data` una vez.

In [ ]:
%matplotlib inline
import importlib.util
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Image, display

EN_COLAB = "google.colab" in sys.modules


def buscar_raiz():
  aqui = Path.cwd().resolve()
  for d in (aqui, *aqui.parents):
    if (d / "src" / "common.py").exists():
      return d
  return None


ROOT = buscar_raiz()


def correr(cmd, cwd=None, ocultar_desde=None):
  # Corre un comando mostrando su salida en vivo (sirve igual en Windows, Linux y Colab).
  # ocultar_desde: deja de imprimir desde la primera línea que empiece así (los JSON largos).
  env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
  cmd = [str(c) for c in cmd]
  p = subprocess.Popen(cmd, cwd=cwd or ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       text=True, encoding="utf-8", errors="replace", bufsize=1)
  oculto = False
  try:
    for linea in p.stdout:
      if ocultar_desde and linea.startswith(ocultar_desde) and not oculto:
        oculto = True
        print("(se omite el JSON completo; queda guardado en results/)")
      if not oculto:
        print(linea.rstrip(chr(13) + chr(10)))
    p.wait()
  except KeyboardInterrupt:
    p.terminate()
    p.wait()
    print("\nInterrumpido. Si era un entrenamiento, volver a correr la celda lo retoma desde last.pt.")
    raise
  if p.returncode:
    raise RuntimeError(f"falló con código {p.returncode}: {' '.join(cmd)}")


if ROOT is None:
  if not EN_COLAB:
    raise RuntimeError("No encuentro la raíz del repo (la carpeta con src/common.py). "
                       "Abrí el notebook desde dentro del repo.")
  os.chdir("/content")
  correr(["git", "clone", "https://github.com/bpeco/ml2-gen-world-model.git"], cwd="/content")
  ROOT = Path("/content/ml2-gen-world-model")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
  sys.path.insert(0, str(ROOT))

if EN_COLAB:
  if not (ROOT / "third_party" / "commavq").exists():
    correr(["git", "clone", "--depth", "1", "https://github.com/commaai/commavq.git", "third_party/commavq"])
  if importlib.util.find_spec("einops") is None or importlib.util.find_spec("lpips") is None:
    correr([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-r", "requirements-metrics.txt"])
  os.environ["HF_HUB_DISABLE_XET"] = "1"  # las descargas con xet se colgaban

if not (ROOT / "data" / "train_tokens.npy").exists():
  if EN_COLAB:
    correr([sys.executable, "-m", "src.data"])
  else:
    raise FileNotFoundError("Faltan los datos (data/train_tokens.npy). Corré `python -m src.data` en la raíz "
                            "del repo y volvé a ejecutar esta celda.")

import torch
from src.common import device_info

print("raíz del repo:", ROOT)
print(device_info())
if torch.cuda.is_available():
  libre, total = torch.cuda.mem_get_info()
  print(f"memoria de GPU: {libre / 2**30:.1f} GB libres de {total / 2**30:.1f} GB")

## 2. Qué hay ya entrenado

`results/checkpoints.json` es el manifiesto de las corridas del informe. **No hace falta reentrenar esas**: se cargan con `src.models.load_model("<run>")` (si el `final.pt` no está en `checkpoints/`, lo baja del almacén compartido con `python -m src.store pull <run>`). La tabla dice cuáles están en esta máquina y cuánto tardó cada entrenamiento (sobre la GPU del informe, una RTX 3080 Ti).

In [ ]:
def leer_json(ruta, defecto=None):
  ruta = Path(ruta)
  return json.loads(ruta.read_text(encoding="utf-8")) if ruta.exists() else defecto


def tabla(cols, filas):
  # tabla HTML simple, sin pandas
  est = "style='text-align:left;padding:2px 12px'"
  cab = "".join(f"<th {est}>{c}</th>" for c in cols)
  cuerpo = "".join("<tr>" + "".join(f"<td {est}>{v}</td>" for v in f) + "</tr>" for f in filas)
  display(HTML(f"<table><tr>{cab}</tr>{cuerpo}</table>"))


manifiesto = {k: v for k, v in leer_json("results/checkpoints.json", {}).items() if not k.startswith("_")}
filas = []
for run, e in sorted(manifiesto.items()):
  tr = leer_json(f"results/train_{run}.json", {})
  local = (ROOT / "checkpoints" / run / "final.pt").exists()
  filas.append([run, e["kind"], f"{e['params'] / 1e6:.1f}", f"{e['tokens'] / 1e6:.0f}",
                json.dumps({k: v for k, v in e["config"].items() if k in ("pos", "p_clean", "mask_schedule")}),
                "sí" if local else "no",
                f"{tr['seconds'] / 60:.1f}" if tr else "-",
                f"{tr['tokens_per_s'] / 1e3:.0f}" if tr else "-"])
if filas:
  tabla(["corrida", "tipo", "params (M)", "tokens (M)", "config", "final.pt local", "entrenamiento (min)", "ktok/s"], filas)
else:
  print("No hay results/checkpoints.json en esta copia del repo: no hay corridas del informe registradas.")

## 3. Parámetros

Todo lo que se puede tocar está en la celda de abajo.

| parámetro | qué es |
|---|---|
| `KIND` | arquitectura: `"gpt"` (autorregresivo), `"maskgit"` (enmascarado, genera en pocos pasos) o `"convlstm"` |
| `SIZE` | tamaño del transformer: `"s"` (~12M params), `"m"` (~27M) o `"l"` (~88M). El ConvLSTM lo ignora |
| `RUN` | nombre de la corrida; se guarda en `CKPT_DIR/RUN/` y todos los resultados llevan este nombre |
| `TOKENS` | presupuesto de entrenamiento en tokens. `20e6` es una demo; el informe usa `100e6` |
| `LR` | learning rate máximo (con warmup y bajada coseno). El informe usa `1e-3` para S y el ConvLSTM, `6e-4` para M |
| `KD`, `ALPHA`, `TAU` | destilación desde `gpt2m` (solo GPT): peso de la pérdida del maestro y temperatura |
| `BS` | batch de ventanas de 20 frames |
| `SEED` | semilla: fija inicialización y orden de los batches |
| `CKPT_DIR` | carpeta de checkpoints (en Colab conviene Drive, porque las sesiones se cortan) |
| `RAPIDO` | `True`: evaluación reducida (5 segmentos de val, 50 clips de rollout). `False`: protocolo completo del informe |
| `PASOS` | pasos de desenmascarado de MaskGIT al generar (1, 2, 4 u 8) |

**Memoria de GPU (aproximada):** S unos 3-4 GB, M unos 8 GB, y la destilación con el maestro `gpt2m` unos 10 GB. Con 12 GB no conviene correr otra cosa pesada a la vez: en Windows, si la GPU se pasa de ~11 GB no da error, pero todo se vuelve 10 a 20 veces más lento.

In [ ]:
KIND = "gpt"            # "gpt" | "maskgit" | "convlstm"
SIZE = "s"              # "s" | "m" | "l"  (el convlstm lo ignora)
RUN = "mi_gpt_s"
TOKENS = 20e6           # demo; el informe usa 100e6
LR = 1e-3               # train_all.sh: s -> 1e-3, m -> 6e-4, convlstm -> 1e-3
KD = False              # destilar desde gpt2m (solo con KIND = "gpt")
ALPHA = 0.5
TAU = 1.0
BS = 16
SEED = 0
CKPT_DIR = "checkpoints"   # en Colab: "/content/drive/MyDrive/ml2/checkpoints" (después de montar Drive)
RAPIDO = True           # evaluación reducida
PASOS = 4               # pasos de MaskGIT al generar

# ---- derivados (no hace falta tocarlos) ----
assert KIND in ("gpt", "maskgit", "convlstm"), KIND
assert not KD or KIND == "gpt", "solo el GPT se destila (MaskGIT y ConvLSTM lo rechazan)"
CONFIG = {"gpt": {"pos": "frame"}, "maskgit": {"pos": "frame", "p_clean": 0.5}, "convlstm": {}}[KIND]  # como train_all.sh
CKPT = Path(CKPT_DIR) / RUN / "final.pt"
PASOS_TOTAL = math.ceil(TOKENS / (BS * 20 * 129))            # 20 frames x 129 tokens por frame
ROLL = f"{RUN}_s{PASOS}" if KIND == "maskgit" else RUN       # nombre de los rollouts
DTYPE, GRAFOS = ("fp32", False) if KIND == "convlstm" else ("fp16", True)   # como eval_all.sh
LIMIT = 50 if RAPIDO else None
SEGMENTOS = 5 if RAPIDO else None
print(f"{KIND} {SIZE if KIND != 'convlstm' else ''} | {RUN} | {TOKENS / 1e6:.0f}M tokens = {PASOS_TOTAL} pasos | config {CONFIG}")

## 4. Entrenar

Corre `python -m src.train` como subproceso y muestra su salida en vivo. Cada línea trae el paso, la pérdida de train (`loss`; con destilación incluye el término del maestro), la `ce` contra los datos reales y la velocidad en tokens por segundo. Cada tanto aparece `[eval]` con la CE sobre ventanas fijas de val reservadas.

- Si `CKPT_DIR/RUN/final.pt` ya existe, la celda avisa y **no reentrena**.
- Si se corta (interrupción, Colab, corte de luz), **volver a correr la celda retoma desde `last.pt`** (se guarda cada 15 minutos). Para retomar hay que dejar `TOKENS`, `LR`, `BS` y `SEED` como estaban.
- El warmup es de 200 pasos como en el informe; en corridas de menos de 2000 pasos se acorta a la décima parte para que el LR llegue a su valor máximo.

In [ ]:
need = {"s": 4, "m": 8, "l": 14}.get(SIZE, 4) if KIND != "convlstm" else 4
need = max(need, 10) if KD else need
if CKPT.exists():
  print(f"{CKPT} ya existe: la corrida '{RUN}' está entrenada, no se reentrena. "
        f"(Para entrenar de nuevo, cambiá RUN o borrá checkpoints/{RUN}.)")
else:
  if torch.cuda.is_available():
    libre = torch.cuda.mem_get_info()[0] / 2**30
    if libre < need + 1:
      print(f"ATENCIÓN: hay {libre:.1f} GB libres y esta corrida necesita unos {need} GB; "
            "si se pasa de ~11 GB en Windows se vuelve mucho más lenta.")
  warmup = 200 if PASOS_TOTAL >= 2000 else max(5, PASOS_TOTAL // 10)
  cmd = [sys.executable, "-u", "-m", "src.train", "--kind", KIND, "--run", RUN, "--tokens", f"{TOKENS:g}",
         "--bs", BS, "--lr", f"{LR:g}", "--warmup", warmup, "--seed", SEED, "--ckpt-dir", CKPT_DIR,
         "--config", json.dumps(CONFIG), "--eval-every", min(250, max(10, PASOS_TOTAL // 10)),
         "--log-every", min(25, max(1, PASOS_TOTAL // 40))]
  if KIND != "convlstm":
    cmd += ["--size", SIZE]
  if KD:
    cmd += ["--kd", "--alpha", ALPHA, "--tau", TAU]
  t0 = time.time()
  correr(cmd)
  print(f"\nentrenamiento terminado en {(time.time() - t0) / 60:.1f} min -> {CKPT}")

## 5. Curva de aprendizaje

Lee `log.jsonl` de la corrida: la pérdida de entrenamiento (cada pocos pasos, línea fina) y la **CE de validación** (`monitor_ce`, sobre 32 ventanas fijas que no se usan para entrenar, línea gruesa con puntos). En gris, la corrida de referencia del mismo tipo y tamaño del informe, si está en esta máquina. Cuando el modelo aprende, ambas curvas bajan juntas; si train baja y val se queda o sube, está memorizando.

La CE de MaskGIT se mide solo sobre tokens enmascarados al azar, así que no es comparable con la de un GPT.

In [ ]:
def leer_log(ruta):
  train, val = {}, {}
  for linea in Path(ruta).read_text().splitlines():
    r = json.loads(linea)
    if "monitor_ce" in r:
      val[r["step"]] = r
    elif "ce" in r:
      train[r["step"]] = r        # si se retomó, los pasos repetidos se quedan con el último
  f = lambda d, k: (np.array([d[s]["tokens"] for s in sorted(d)]) / 1e6, np.array([d[s][k] for s in sorted(d)]))
  return f(train, "ce"), f(val, "monitor_ce")


REF = "convlstm" if KIND == "convlstm" else f"{KIND}_{SIZE}"
ref_log = next((p for p in (Path(CKPT_DIR) / REF / "log.jsonl", ROOT / "checkpoints" / REF / "log.jsonl")
                if p.exists() and REF != RUN), None)
AZUL, GRIS = "#2a78d6", "#9a9a9a"
(tr, ce_tr), (va, ce_va) = leer_log(Path(CKPT_DIR) / RUN / "log.jsonl")

fig, ax = plt.subplots(figsize=(8, 4.2))
if ref_log:
  (rtr, rce_tr), (rva, rce_va) = leer_log(ref_log)
  ax.plot(rtr, rce_tr, color=GRIS, lw=1, alpha=0.5, label=f"{REF} (referencia), train")
  ax.plot(rva, rce_va, color=GRIS, lw=2, marker="s", ms=4, label=f"{REF} (referencia), val: {rce_va[-1]:.3f}")
ax.plot(tr, ce_tr, color=AZUL, lw=1, alpha=0.5, label=f"{RUN}, train")
ax.plot(va, ce_va, color=AZUL, lw=2, marker="o", ms=4, label=f"{RUN}, val: {ce_va[-1]:.3f}")
ax.set_xlabel("tokens vistos (millones)")
ax.set_ylabel("cross-entropy (nats por token)")
ax.set_title("Curva de aprendizaje")
ax.grid(alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.show()
if ref_log:
  a_mismos = np.interp(va[-1], rva, rce_va)
  print(f"CE de val de {RUN}: {ce_va[-1]:.3f} con {va[-1]:.0f}M tokens. "
        f"La referencia {REF} tenía {a_mismos:.3f} a esos mismos tokens y {rce_va[-1]:.3f} al final ({rva[-1]:.0f}M).")

## 6. Evaluar la CE

`src.evaluate` aplica el protocolo de comma: ventanas de 20 frames sobre los 100 segmentos de val. `ce_all` es la CE por token incluyendo los BOS (comparable con el 2,02 que reporta comma para su `gpt2m`) y `ce_last` es la CE solo de los 128 códigos del último frame de cada ventana, que es lo que mide un modelo cuando *genera*. MaskGIT no tiene verosimilitud, así que solo tiene `ce_last`.

Con `RAPIDO` se usan solo los 5 primeros segmentos; las referencias se recalculan sobre esos mismos segmentos cuando guardaron los valores por segmento (`gpt_s` sí; `maskgit_s`, `convlstm` y `gpt2m` solo traen el promedio de los 100, y la columna `segmentos` lo dice). Con 5 segmentos la CE es ruidosa: sirve para ver el orden de magnitud.

In [ ]:
cmd = [sys.executable, "-u", "-m", "src.evaluate", "--model", CKPT, "--name", RUN]
if SEGMENTOS:
  cmd += ["--segments", SEGMENTOS]
correr(cmd, ocultar_desde="{")

sufijo = f"_seg{SEGMENTOS}" if SEGMENTOS else ""
mi_eval = leer_json(f"results/eval_{RUN}{sufijo}.json")
n = mi_eval["segments"]


def ce_de(ruta):
  # (ce_all, ce_last, segmentos). Las referencias viejas no guardan per_segment: se usan sus promedios de val completo.
  r = leer_json(ruta)
  if r is None:
    return None
  ps = r.get("per_segment")
  if ps and n <= r["segments"]:
    m = lambda k: float(np.mean(ps[k][:n])) if k in ps else None
    return m("ce_all"), m("ce_last"), n
  return r["ce_all"], r["ce_last"], r["segments"]


filas, barras = [], []
fmt = lambda v: "-" if v is None else f"{v:.3f}"
for nombre, ruta in [(RUN + " (nuevo)", f"results/eval_{RUN}{sufijo}.json"), ("gpt_s", "results/eval_gpt_s.json"),
                     ("maskgit_s", "results/eval_maskgit_s.json"), ("convlstm", "results/eval_convlstm.json"),
                     ("gpt2m (comma)", "results/eval_gpt2m.json")]:
  if nombre == RUN:
    continue
  v = ce_de(ruta)
  if v:
    filas.append([nombre, fmt(v[0]), fmt(v[1]), v[2]])
    barras.append((nombre, v[1]))
tabla(["modelo", "ce_all", "ce_last", "segmentos"], filas)
print("Más bajo es mejor. Si una referencia usa más segmentos que el nuevo, es porque no guardó los valores por segmento.")

fig, ax = plt.subplots(figsize=(7, 0.45 * len(barras) + 0.8))
ax.barh([b[0] for b in barras][::-1], [b[1] for b in barras][::-1],
        color=[AZUL if "(nuevo)" in b[0] else GRIS for b in barras][::-1])
ax.set_xlabel("ce_last (nats por token, último frame)")
ax.grid(axis="x", alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 7. Rollouts y métricas de video

`src.rollouts` toma los clips fijos del informe (1 s de contexto real) y genera 3 s con el modelo, en fp16 y con CUDA graphs como en `eval_all.sh` (el ConvLSTM va en fp32 y sin graphs). Guarda los tokens en `results/rollouts/<nombre>.npy`. Con `RAPIDO` son 50 clips en vez de 400.

Después `src.metrics_video` decodifica los tokens a imágenes y calcula **LPIPS** (parecido perceptual frame a frame con el video real; más bajo es mejor) y **FVD** (distancia entre la distribución de videos reales y generados, ventanas que terminan a 1, 2 y 3 s; más bajo es mejor).

Las métricas sí aceptan rollouts parciales, pero la primera vez que se corren tienen que decodificar los 400 clips reales de referencia y guardar su caché (~2,4 GB en `results/cache`), y eso tarda. Por eso, en modo `RAPIDO` **se saltean si esa caché no existe**; con `RAPIDO = False` la crean. El FVD con pocos clips es muy ruidoso y su valor depende de cuántos clips hay: las referencias de la tabla se recalculan con esos mismos clips.

In [ ]:
cmd = [sys.executable, "-u", "-m", "src.rollouts", "--model", CKPT, "--name", ROLL, "--dtype", DTYPE]
if GRAFOS:
  cmd += ["--graphs"]
if KIND == "maskgit":
  cmd += ["--steps", PASOS]
if LIMIT:
  cmd += ["--limit", LIMIT]
t0 = time.time()
correr(cmd)
print(f"rollouts en {time.time() - t0:.0f} s")

In [ ]:
cache = ROOT / "results" / "cache"
hay_cache = any(cache.glob("real_frames_*.ok"))
faltan = [m for m in ("lpips", "cdfvd") if importlib.util.find_spec(m) is None]
if faltan:
  print("Faltan paquetes para las métricas de video:", faltan, "-> pip install -r requirements-metrics.txt")
elif RAPIDO and not hay_cache:
  print("RAPIDO: se saltean LPIPS y FVD porque no existe la caché de referencia en results/cache "
        "(crearla decodifica los 400 clips reales y tarda). Poné RAPIDO = False para calcularlas.")
else:
  cmd = [sys.executable, "-u", "-m", "src.metrics_video", "--name", ROLL]
  if RAPIDO:
    cmd += ["--boot", 200, "--boot-fvd", 50]
  t0 = time.time()
  correr(cmd)
  print(f"métricas en {time.time() - t0:.0f} s")

In [ ]:
mi_video = leer_json(f"results/video_{ROLL}.json")
if mi_video is None:
  print("Todavía no hay métricas de video de esta corrida (ver la celda anterior).")
else:
  nc = mi_video["n_clips"]
  from src import metrics_video as mv
  h = mv.clips_hash()
  real_f = np.load(cache / f"fvd_real_{mi_video['fvd_model']}_{h}.npz")

  def video_de(nombre):
    # LPIPS y FVD de un rollout guardado, sobre sus primeros nc clips
    try:
      lp = np.load(cache / f"lpips_{nombre}.npy")[:nc]
      ff = np.load(cache / f"fvd_fake_{mi_video['fvd_model']}_{nombre}.npz")
    except FileNotFoundError:
      return None
    return ([float(lp[:, mv.AT[k]].mean()) for k in ("1", "2", "3")],
            [mv.fvd_from_feats(real_f[k][:nc], ff[k][:nc]) for k in ("1", "2", "3")])

  ic = lambda d: f"{d['value']:.0f} [{d['lo']:.0f}, {d['hi']:.0f}]"
  filas = [[RUN + " (nuevo)"] + [f"{mi_video['lpips']['at'][k]['mean']:.3f}" for k in ("1", "2", "3")]
           + [ic(mi_video["fvd"][k]) for k in ("1", "2", "3")]]
  for ref in ("gpt2m", "gpt_s", "maskgit_s_s4", "convlstm"):
    if ref != ROLL and (v := video_de(ref)):
      filas.append([ref] + [f"{x:.3f}" for x in v[0]] + [f"{x:.0f}" for x in v[1]])
  tabla(["modelo", "LPIPS 1 s", "LPIPS 2 s", "LPIPS 3 s", "FVD 1 s", "FVD 2 s", "FVD 3 s"], filas)
  print(f"{nc} clips. Entre corchetes, intervalo de confianza del 95%. Más bajo es mejor.")
  if nc < 400:
    print("Con tan pocos clips el FVD es muy ruidoso: sirve para ver el orden de magnitud, no para comparar de a poco.")

## 8. Ver muestras

Para algunos clips del set fijo se muestra una grilla (filas = modelos, columnas = el último frame de contexto y lo generado a +0,5, +1, +2 y +3 s) y un GIF con todo lado a lado. Los clips se eligen por cuánto se mueve la escena real en 3 s (percentiles 50, 75 y 90), no por cómo le va al modelo. Se compara el video **real**, el `gpt2m` de comma (si sus rollouts están en `results/rollouts/`) y el modelo nuevo. Esto necesita el decoder del VQ-VAE (`data/raw/`) y `third_party/commavq`.

In [ ]:
from src import ejemplos

gen = np.load(ROOT / "results" / "rollouts" / f"{ROLL}.npy")        # (clips, 60, 128)
n = len(gen)
modelos = ["real"] + (["gpt2m"] if (ejemplos.ROLL / "gpt2m.npy").exists() else [])
try:
  orden = np.argsort(ejemplos.cambio_escena()[:n])
  clips = [int(orden[int(round(p / 100 * (n - 1)))]) for p in ejemplos.PERCENTILES]
except FileNotFoundError:
  clips = [0, n // 2, n - 1]
  print("No está results/cache/lpips_frozen.npy: se usan clips fijos en vez de percentiles de movimiento.")

for i, c in enumerate(clips):
  ctx_img, imgs = ejemplos.clip(c, modelos, generados={RUN: np.asarray(gen[c]).astype(np.int64)})
  try:
    titulo, _ = ejemplos.titulo_clip(c)
  except FileNotFoundError:
    titulo = f"Clip {c}: 1 s de contexto real y 3 s generados a temperatura 1"
  fig = ejemplos.grilla(ctx_img, imgs, titulo)
  display(fig)
  plt.close(fig)
  if i == 0:
    display(Image(data=ejemplos.gif(ctx_img, imgs), format="gif"))

## 9. Latencia

`src.bench` mide cuánto tarda el modelo en generar **un frame** con batch 1 (mediana de 100 frames después de calentar), sin contar el decoder. Para manejar en tiempo real hacen falta menos de 50 ms por frame (20 fps). **Hay que correrlo con la GPU libre**: si hay otro proceso usando la GPU (otro entrenamiento, otro notebook) la medida sale inflada. La latencia no depende de los pesos, solo de la arquitectura.

In [ ]:
cmd = [sys.executable, "-u", "-m", "src.bench", "--model", CKPT, "--name", RUN, "--dtype", DTYPE]
if GRAFOS:
  cmd += ["--graphs"]
if KIND == "maskgit":
  cmd += ["--steps", PASOS]
correr(cmd)

b = max((ROOT / "results" / "bench").glob(f"{RUN}__*.json"), key=lambda p: p.stat().st_mtime)
r = json.loads(b.read_text())
print(f"\n{RUN}: {r['median_ms']:.2f} ms por frame (mediana)  ->  {r['fps']:.0f} fps  "
      f"({'sí' if r['realtime'] else 'no'} llega a tiempo real, 20 fps)")
if torch.cuda.is_available():
  print(f"GPU: {r['device'].get('gpu')} | otros procesos en la GPU distorsionan esta medida.")

refs = [("gpt_s", "gpt_s__fp16_graphs"), ("maskgit_s, 4 pasos", "maskgit_s__fp16_graphs_s4"),
        ("convlstm", "convlstm__fp32"), ("gpt2m (comma)", "gpt2m__fp16_graphs")]
filas = [[RUN + " (nuevo)", f"{r['median_ms']:.2f}", f"{r['fps']:.0f}"]]
for nombre, f in refs:
  rr = leer_json(f"results/bench/{f}.json")
  if rr:
    filas.append([nombre + " (informe)", f"{rr['median_ms']:.2f}", f"{rr['fps']:.0f}"])
tabla(["modelo", "ms por frame", "fps"], filas)

## 10. Guardar y compartir

Las celdas de abajo están comentadas porque cambian archivos del repo o suben cosas a otro lado; descomentá lo que quieras usar.

- `register` anota la corrida en `results/checkpoints.json` (el manifiesto que va en git): su sha256, tamaño, config y tokens. Necesita `checkpoints/<RUN>/final.pt` y `results/train_<RUN>.json`, así que solo funciona con `CKPT_DIR = "checkpoints"`.
- `push` sube el checkpoint (y con `--rollouts` los rollouts) a una carpeta compartida (`--dir`, por ejemplo en Drive) o a un repo de modelos de Hugging Face (`--hf usuario/repo`, con `--create` la primera vez; necesita `hf auth login` o `HF_TOKEN`). Nada se sube sin este comando.
- Quien quiera usar la corrida la baja con `python -m src.store pull <RUN> --dir ...` y la carga con `load_model("<RUN>")`.

In [ ]:
# correr([sys.executable, "-m", "src.store", "register", RUN])
# correr([sys.executable, "-m", "src.store", "push", RUN, "--dir", "/content/drive/MyDrive/ml2/store", "--rollouts"])
# correr([sys.executable, "-m", "src.store", "push", RUN, "--hf", "usuario/ml2-wm", "--create"])
# correr([sys.executable, "-m", "src.store", "list"])

### Agregar la corrida al reporte

El reporte (`python -m src.report`) lee `results/configs.json`, que arma `scripts/make_configs.py` con una **lista fija** de configuraciones. Para que aparezca esta corrida hay que sumar una entrada ahí, por ejemplo junto a las de GPT:

```python
add(entry("Mi GPT-S", "gpt", R / "eval_mi_gpt_s.json", R / "video_mi_gpt_s.json",
          R / "bench/mi_gpt_s__fp16_graphs.json", params("mi_gpt_s"), autoregressive=True, short="Mi GPT-S"))
```

(Cambiar los nombres por los de `RUN`/`ROLL`; para MaskGIT el video es `video_<RUN>_s<PASOS>.json` y el bench `bench/<RUN>__fp16_graphs_s<PASOS>.json`.) Después: `python scripts/make_configs.py && python -m src.report`. Para que sea comparable con las demás hay que haber evaluado con `RAPIDO = False`.